In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.utils.random import sample_without_replacement
from scipy import io
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import linear_model as lm

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

In [ ]:
power,coherence,granger,labels = load_data('/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_all_validate3.mat',
                                          fBounds=(1,56),feature_list=['power','coherence','granger'])

power = 10*power
power = power.astype(np.float32)
power[power>6] = 6 

coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X = np.hstack((power,coherence,granger))


In [ ]:
windows = labels['windows']
mouse = np.squeeze(windows['mouse'])
expDate = np.squeeze(windows['expDate'])
group = np.squeeze(windows['group'])
condition = np.squeeze(windows['condition'])
behavior = np.squeeze(windows['behavior'])
time = np.squeeze(windows['time'])

In [ ]:
idx_pos = (condition==4)&(behavior==1)
indx_neg = (behavior==2)&((condition==4)|(condition==6)|(condition==8))
y = np.zeros(len(mouse))
y[idx_pos] = 1
idx_tot = idx_pos|indx_neg

In [ ]:
mice = np.unique(mouse)
nMice = len(mice)

## Massive function for analysis

In [ ]:
def analysis(nTrain):
    myDict = {}
    m_idx = rand.choice(len(mice),nTrain,replace=False)
    tr_idx = np.zeros(nMice)
    tr_idx[m_idx] = 1
    mice_train = mice[tr_idx==1]
    mice_test = mice[tr_idx==0]
    myDict['mice_train'] = m_idx
    
    training = np.zeros(len(mouse))
    for i in range(nTrain):
        training[mouse==mice_train[i]] = 1
    
    X_train = X[training==1]
    X_test = X[training==0]
    
    model_nmf = dp.NMF(30)
    S_train = model_nmf.fit_transform(X_train)
    
    Ex = np.mean(X_train,axis=0)
    myDict['recon_random_train'] = np.mean((X_train-Ex)**2)
    myDict['recon_random_test_p'] = np.mean((X_test-Ex)**2)
    Ex = np.mean(X_test,axis=0)
    myDict['recon_random_test'] = np.mean((X_test-Ex)**2)
    X_recon = np.dot(S_train,model_nmf.components_)
    myDict['recon_train'] = np.mean((X_train-X_recon)**2)
    S_test = model_nmf.transform(X_test)
    X_recon = np.dot(S_test,model_nmf.components_)
    myDict['recon_test'] = np.mean((X_test-X_recon)**2)
    
    idx_tot_train = idx_tot[training==1]
    S_sub_train = S_train[idx_tot_train]
    y_train = y[training==1]
    y_sub_train = y_train[idx_tot_train]
    
    idx_tot_test = idx_tot[training==0]
    S_sub_test = S_test[idx_tot_test]
    y_test = y[training==0]
    y_sub_test = y_test[idx_tot_test]
    
    model_lm = lm.LogisticRegressionCV()
    model_lm.fit(S_sub_train,y_sub_train)
    
    y_pred = model_lm.decision_function(S_sub_train)
    myDict['overall_auc_train'] = roc_auc_score(y_sub_train,y_pred)  
    
    y_pred = model_lm.decision_function(S_sub_test)
    myDict['overall_auc'] = roc_auc_score(y_sub_test,y_pred)
    
    mouse_test = mouse[training==0]
    mouse_sub_test = mouse_test[idx_tot_test] 
    
    test_auc_list = np.zeros(8-nTrain)
    train_auc_list = np.zeros(nTrain)
    
    for i in range(8-nTrain):
        yp = y_pred[mouse_sub_test==mice_test[i]]
        yt = y_sub_test[mouse_sub_test==mice_test[i]]
        test_auc_list[i] = roc_auc_score(yt,yp)
    
    y_pred = model_lm.decision_function(S_sub_train)
    mouse_train = mouse[training==1]
    mouse_sub_train = mouse_train[idx_tot_train] 
    
    for i in range(nTrain):
        yp = y_pred[mouse_sub_train==mice_train[i]]
        yt = y_sub_train[mouse_sub_train==mice_train[i]]
        train_auc_list[i] = roc_auc_score(yt,yp)   
    
    myDict['mouse_auc_train'] = train_auc_list
    myDict['mouse_auc_test'] = test_auc_list
    return myDict

In [ ]:
trial_1 = analysis(4)

In [ ]:
print(trial_1['mouse_auc_train'],trial_1['mouse_auc_test'])
print(trial_1['overall_auc_train'],trial_1['overall_auc'])

In [ ]:
trial_2 = analysis(4)
trial_3 = analysis(4)
trial_4 = analysis(4)
trial_5 = analysis(4)
trial_6 = analysis(4)
trial_7 = analysis(4)
trial_8 = analysis(4)
trial_9 = analysis(4)

In [ ]:
print(np.mean(trial_2['mouse_auc_test']))
print(np.mean(trial_3['mouse_auc_test']))
print(np.mean(trial_4['mouse_auc_test']))
print(np.mean(trial_5['mouse_auc_test']))
print(np.mean(trial_6['mouse_auc_test']))
print(np.mean(trial_7['mouse_auc_test']))
print(np.mean(trial_8['mouse_auc_test']))
print(np.mean(trial_9['mouse_auc_test']))

In [ ]:
a1 = np.zeros(9)
a1[0] = np.mean(trial_1['mouse_auc_test'])
a1[1] = np.mean(trial_2['mouse_auc_test'])
a1[2] = np.mean(trial_3['mouse_auc_test'])
a1[3] = np.mean(trial_4['mouse_auc_test'])
a1[4] = np.mean(trial_5['mouse_auc_test'])
a1[5] = np.mean(trial_6['mouse_auc_test'])
a1[6] = np.mean(trial_7['mouse_auc_test'])
a1[7] = np.mean(trial_8['mouse_auc_test'])
a1[8] = np.mean(trial_9['mouse_auc_test'])
print(np.mean(a1),np.std(a1)*1.96/3)

In [ ]:
print(np.mean(trial_2['mouse_auc_train']))
print(np.mean(trial_3['mouse_auc_train']))
print(np.mean(trial_4['mouse_auc_train']))
print(np.mean(trial_5['mouse_auc_train']))
print(np.mean(trial_6['mouse_auc_train']))
print(np.mean(trial_7['mouse_auc_train']))
print(np.mean(trial_8['mouse_auc_train']))
print(np.mean(trial_9['mouse_auc_train']))

In [ ]:
trial_16 = analysis(6)
trial_26 = analysis(6)
trial_36 = analysis(6)
trial_46 = analysis(6)
trial_56 = analysis(6)
trial_66 = analysis(6)
trial_76 = analysis(6)

In [ ]:
print(np.mean(trial_16['mouse_auc_test']))
print(np.mean(trial_26['mouse_auc_test']))
print(np.mean(trial_36['mouse_auc_test']))
print(np.mean(trial_46['mouse_auc_test']))
print(np.mean(trial_56['mouse_auc_test']))
print(np.mean(trial_66['mouse_auc_test']))
print(np.mean(trial_76['mouse_auc_test']))

In [ ]:
print(np.mean(trial_16['mouse_auc_train']))
print(np.mean(trial_26['mouse_auc_train']))
print(np.mean(trial_36['mouse_auc_train']))
print(np.mean(trial_46['mouse_auc_train']))
print(np.mean(trial_56['mouse_auc_train']))
print(np.mean(trial_66['mouse_auc_train']))
print(np.mean(trial_76['mouse_auc_train']))